# Generate CO2 Pipeline and Injection CSVs for China

Creates `co2_pipeline.csv` and `co2_injection.csv` for a 31-province × 24-basin model.

- **Pipeline**: 744 rows — each province can ship CO2 to any of 24 basins
- **Injection**: 744 rows — one injection asset per province-basin pair

Cost parameters:
- Pipeline investment cost: **0.0510 M\$/(Mt/yr/km)** — MACRO multiplies by distance internally
- Transportation (variable OM): **0.022 \$/(t·km)** × distance
- Injection capital cost: **18 M\$/(Mt/yr)**
- Storage (variable OM): **2.93 \$/t**

In [2]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import os

# Province capital coordinates (lat, lon)
CAPITALS = {
    "Beijing":        (39.9042, 116.4074),
    "Tianjin":        (39.3434, 117.3616),
    "Hebei":          (38.0428, 114.5149),
    "Shanxi":         (37.8706, 112.5489),
    "Inner Mongolia": (40.8175, 111.7655),
    "Liaoning":       (41.8057, 123.4315),
    "Jilin":          (43.8868, 125.3245),
    "Heilongjiang":   (45.8038, 126.5349),
    "Shanghai":       (31.2304, 121.4737),
    "Jiangsu":        (32.0603, 118.7969),
    "Zhejiang":       (30.2741, 120.1551),
    "Anhui":          (31.8612, 117.2830),
    "Fujian":         (26.0745, 119.2965),
    "Jiangxi":        (28.6820, 115.8579),
    "Shandong":       (36.6512, 117.1201),
    "Henan":          (34.7466, 113.6254),
    "Hubei":          (30.5928, 114.3055),
    "Hunan":          (28.2278, 112.9388),
    "Guangdong":      (23.1291, 113.2644),
    "Guangxi":        (22.8170, 108.3665),
    "Hainan":         (20.0440, 110.3419),
    "Chongqing":      (29.5630, 106.5516),
    "Sichuan":        (30.5723, 104.0665),
    "Guizhou":        (26.5983, 106.7072),
    "Yunnan":         (25.0453, 102.7097),
    "Tibet":          (29.6520,  91.1721),
    "Shaanxi":        (34.2658, 108.9541),
    "Gansu":          (36.0611, 103.8343),
    "Qinghai":        (36.6232, 101.7782),
    "Ningxia":        (38.4872, 106.2309),
    "Xinjiang":       (43.7928,  87.6271),
}

# MACRO region codes matching existing system nodes
PROVINCE_CODES = {
    "Beijing":        "Region1Beijing",
    "Tianjin":        "Region2Tianjin",
    "Hebei":          "Region3Hebei",
    "Shanxi":         "Region4Shanxi",
    "Inner Mongolia": "Region5Innermongolia",
    "Liaoning":       "Region6Liaoning",
    "Jilin":          "Region7Jilin",
    "Heilongjiang":   "Region8Heilongjiang",
    "Shanghai":       "Region9Shanghai",
    "Jiangsu":        "Region10Jiangsu",
    "Zhejiang":       "Region11Zhejiang",
    "Anhui":          "Region12Anhui",
    "Fujian":         "Region13Fujian",
    "Jiangxi":        "Region14Jiangxi",
    "Shandong":       "Region15Shandong",
    "Henan":          "Region16Henan",
    "Hubei":          "Region17Hubei",
    "Hunan":          "Region18Hunan",
    "Guangdong":      "Region19Guangdong",
    "Guangxi":        "Region20Guangxi",
    "Hainan":         "Region21Hainan",
    "Chongqing":      "Region22Chongqing",
    "Sichuan":        "Region23Sichuan",
    "Guizhou":        "Region24Guizhou",
    "Yunnan":         "Region25Yunnan",
    "Tibet":          "Region26Tibet",
    "Shaanxi":        "Region27Shaanxi",
    "Gansu":          "Region28Gansu",
    "Qinghai":        "Region29Qinghai",
    "Ningxia":        "Region30Ningxia",
    "Xinjiang":       "Region31Xinjiang",
}

# Short keys for vertex names (no spaces or special chars)
BASIN_KEYS = {
    "Songliao Basin":             "Songliao",
    "Turpan-Hami Basin":          "TurpanHami",
    "Subei Basin":                "Subei",
    "Bohai Bay Basin (onshore)":  "BohaiOnshore",
    "Qaidam Basin":               "Qaidam",
    "Nanxiang Basin":             "Nanxiang",
    "Sanjiang Basin":             "Sanjiang",
    "Hailar Basin":               "Hailar",
    "Jianghan Basin":             "Jianghan",
    "Tarim Basin":                "Tarim",
    "Ordos Basin":                "Ordos",
    "Yingen-Ejina Basin":         "YingenEjina",
    "Hehuai Basin":               "Hehuai",
    "Qinshui Basin":              "Qinshui",
    "Erlian Basin":               "Erlian",
    "Junggar Basin":              "Junggar",
    "Sichuan Basin":              "SichuanBasin",
    "Bohai Bay Basin (offshore)": "BohaiOffshore",
    "North Yellow Sea Basin":     "NorthYellowSea",
    "South Yellow Sea Basin":     "SouthYellowSea",
    "East China Sea Basin":       "EastChinaSea",
    "Pearl River Mouth Basin":    "PearlRiverMouth",
    "Beibu Gulf Basin":           "BeibugGulf",
    "Qiongdongnan Basin":         "Qiongdongnan",
}

# Cost parameters
PIPELINE_INVEST_COST  = 0.0510 * 8760  # M$/(Mt/h) per km
TRANSPORT_VAR_COST    = 0.022   # $/(t*km)
INJECTION_INVEST_COST = 18.0 * 8760 # M$/(Mt/h)
STORAGE_VAR_COST      = 2.93    # $/t

print(f"Provinces: {len(CAPITALS)}, Basins: {len(BASIN_KEYS)}")
print(f"Expected pipeline/injection rows: {len(CAPITALS) * len(BASIN_KEYS)}")

Provinces: 31, Basins: 24
Expected pipeline/injection rows: 744


In [3]:
# Load basin centroids
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('make_ccs_json_files.ipynb'))
BASIN_CSV = os.path.join(NOTEBOOK_DIR, 'usgs_source', 'basin_centroids_shapefile.csv')

basins_raw = pd.read_csv(
    BASIN_CSV,
    names=['idx','basin','type','lon','lat','source','shapefile','notes'],
    skiprows=1
)
basins_df = basins_raw[['basin','lat','lon']].copy()

missing = set(BASIN_KEYS.keys()) - set(basins_df['basin'])
if missing:
    raise ValueError(f"Basins missing from CSV: {missing}")

basins_df = basins_df[basins_df['basin'].isin(BASIN_KEYS)].copy()
basins_df['key'] = basins_df['basin'].map(BASIN_KEYS)
basins_df = basins_df.set_index('basin')
print(basins_df[['key','lat','lon']].to_string())

                                        key      lat       lon
basin                                                         
Songliao Basin                     Songliao  45.5424  124.2489
Turpan-Hami Basin                TurpanHami  42.7741   91.8474
Subei Basin                           Subei  33.2802  120.2968
Bohai Bay Basin (onshore)      BohaiOnshore  38.1531  117.0270
Qaidam Basin                         Qaidam  37.4565   94.0675
Nanxiang Basin                     Nanxiang  32.6284  112.3891
Sanjiang Basin                     Sanjiang  47.0000  132.0000
Hailar Basin                         Hailar  48.9011  118.5403
Jianghan Basin                     Jianghan  30.4742  112.9206
Tarim Basin                           Tarim  39.3991   82.2819
Ordos Basin                           Ordos  37.5548  108.7178
Yingen-Ejina Basin              YingenEjina  41.5000  101.5000
Hehuai Basin                         Hehuai  34.0000  115.0000
Qinshui Basin                       Qinshui  36.7143  1

In [4]:
# Haversine distance in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Build all province-basin distance pairs
records = []
for prov, (cap_lat, cap_lon) in CAPITALS.items():
    pcode = PROVINCE_CODES[prov]
    for basin_name, row in basins_df.iterrows():
        dist = haversine(cap_lat, cap_lon, row['lat'], row['lon'])
        records.append({
            'province':    prov,
            'pcode':       pcode,
            'basin':       basin_name,
            'basin_key':   row['key'],
            'distance_km': round(dist, 1),
        })

dist_df = pd.DataFrame(records)
print(f"Total province-basin pairs: {len(dist_df)}")
print(f"Distance range: {dist_df['distance_km'].min():.1f} – {dist_df['distance_km'].max():.1f} km")
print("\nNearest basin per province:")
nearest = dist_df.loc[dist_df.groupby('province')['distance_km'].idxmin(),
                      ['province','basin','distance_km']]
print(nearest.to_string(index=False))

Total province-basin pairs: 744
Distance range: 104.3 – 3989.1 km

Nearest basin per province:
      province                     basin  distance_km
         Anhui              Hehuai Basin        319.3
       Beijing Bohai Bay Basin (onshore)        201.9
     Chongqing             Sichuan Basin        104.3
        Fujian      East China Sea Basin        647.5
         Gansu               Ordos Basin        465.3
     Guangdong   Pearl River Mouth Basin        310.9
       Guangxi          Beibu Gulf Basin        285.3
       Guizhou             Sichuan Basin        424.0
        Hainan          Beibu Gulf Basin        128.9
         Hebei Bohai Bay Basin (onshore)        220.2
  Heilongjiang            Songliao Basin        180.0
         Henan              Hehuai Basin        151.0
         Hubei            Jianghan Basin        133.3
         Hunan            Jianghan Basin        249.8
Inner Mongolia               Ordos Basin        447.8
       Jiangsu               Subei Basin 

In [5]:
# Generate co2_pipeline.csv  (31 x 24 = 744 rows)
pipeline_rows = []
for _, r in dist_df.iterrows():
    pcode     = r['pcode']
    bkey      = r['basin_key']
    dist      = r['distance_km']
    var_om    = round(TRANSPORT_VAR_COST * dist, 4)
    start_vtx = f"co2_captured_{pcode}"
    end_vtx   = f"co2_transported_{pcode}_to_{bkey}"

    pipeline_rows.append({
        'Type':                                                  'OneWayTransmissionLink',
        'id':                                                    f"{pcode}_to_{bkey}_CO2_Pipeline",
        'location':                                              pcode,
        'edges--transmission_edge--commodity':                   'CO2Captured',
        'edges--transmission_edge--has_capacity':                'TRUE',
        'edges--transmission_edge--can_expand':                  'TRUE',
        'edges--transmission_edge--can_retire':                  'FALSE',
        'edges--transmission_edge--integer_decisions':           'FALSE',
        'edges--transmission_edge--constraints--CapacityConstraint': 'TRUE',
        'edges--transmission_edge--start_vertex':                start_vtx,
        'edges--transmission_edge--end_vertex':                  end_vtx,
        'edges--transmission_edge--distance':                    dist,
        'edges--transmission_edge--existing_capacity':           0,
        'edges--transmission_edge--investment_cost':             PIPELINE_INVEST_COST,
        'edges--transmission_edge--fixed_om_cost':               0,
        'edges--transmission_edge--variable_om_cost':            var_om,
        'edges--transmission_edge--wacc':                        0.08,
        'edges--transmission_edge--lifetime':                    25,
        'edges--transmission_edge--capital_recovery_period':     15,
        'edges--transmission_edge--loss_fraction':               0.0,
    })

pipeline_df = pd.DataFrame(pipeline_rows)
out_pipeline = os.path.join(NOTEBOOK_DIR, 'co2_pipeline.csv')
pipeline_df.to_csv(out_pipeline, index=False)
print(f"Wrote {len(pipeline_df)} rows -> {out_pipeline}")
pipeline_df.head(3)

Wrote 744 rows -> /Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/co2_basin_analysis/co2_pipeline.csv


,Type,id,location,edges--transmission_edge--commodity,edges--transmission_edge--has_capacity,edges--transmission_edge--can_expand,edges--transmission_edge--can_retire,edges--transmission_edge--integer_decisions,edges--transmission_edge--constraints--CapacityConstraint,edges--transmission_edge--start_vertex,edges--transmission_edge--end_vertex,edges--transmission_edge--distance,edges--transmission_edge--existing_capacity,edges--transmission_edge--investment_cost,edges--transmission_edge--fixed_om_cost,edges--transmission_edge--variable_om_cost,edges--transmission_edge--wacc,edges--transmission_edge--lifetime,edges--transmission_edge--capital_recovery_period,edges--transmission_edge--loss_fraction
0,OneWayTransmissionLink,Region1Beijing_to_Songliao_CO2_Pipeline,Region1Beijing,CO2Captured,TRUE,TRUE,FALSE,FALSE,TRUE,co2_captured_Region1Beijing,co2_transported_Region1Beijing_to_Songliao,895.5,0,446.76,0,19.7010,0.08,25,15,0.0
1,OneWayTransmissionLink,Region1Beijing_to_TurpanHami_CO2_Pipeline,Region1Beijing,CO2Captured,TRUE,TRUE,FALSE,FALSE,TRUE,co2_captured_Region1Beijing,co2_transported_Region1Beijing_to_TurpanHami,2067.6,0,446.76,0,45.4872,0.08,25,15,0.0
2,OneWayTransmissionLink,Region1Beijing_to_Subei_CO2_Pipeline,Region1Beijing,CO2Captured,TRUE,TRUE,FALSE,FALSE,TRUE,co2_captured_Region1Beijing,co2_transported_Region1Beijing_to_Subei,814.1,0,446.76,0,17.9102,0.08,25,15,0.0


In [6]:
# Generate co2_injection.csv  (31 x 24 = 744 rows)
injection_rows = []
for _, r in dist_df.iterrows():
    pcode     = r['pcode']
    bkey      = r['basin_key']
    start_vtx = f"co2_transported_{pcode}_to_{bkey}"
    end_vtx   = f"co2_storage_{bkey}"

    injection_rows.append({
        'Type':                                                     'CO2Injection',
        'id':                                                       f"{pcode}_to_{bkey}_CO2_Injection",
        'location':                                                 pcode,
        'edges--co2_captured_edge--integer_decisions':              'FALSE',
        'edges--co2_captured_edge--can_retire':                     'FALSE',
        'edges--co2_captured_edge--constraints--CapacityConstraint': 'TRUE',
        'edges--co2_captured_edge--unidirectional':                 'TRUE',
        'edges--co2_captured_edge--commodity':                      'CO2Captured',
        'edges--co2_captured_edge--can_expand':                     'TRUE',
        'edges--co2_captured_edge--uc':                             'FALSE',
        'edges--co2_captured_edge--has_capacity':                   'TRUE',
        'edges--co2_storage_edge--unidirectional':                  'TRUE',
        'edges--co2_storage_edge--commodity':                       'CO2Captured',
        'edges--co2_storage_edge--has_capacity':                    'FALSE',
        'transforms--constraints--BalanceConstraint':               'TRUE',
        'transforms--timedata':                                     'CO2Captured',
        'edges--co2_captured_edge--investment_cost':                INJECTION_INVEST_COST,
        'edges--co2_captured_edge--fixed_om_cost':                  0,
        'edges--co2_captured_edge--variable_om_cost':               STORAGE_VAR_COST,
        'edges--co2_captured_edge--wacc':                           0.08,
        'edges--co2_captured_edge--lifetime':                       40,
        'edges--co2_captured_edge--capital_recovery_period':        20,
        'edges--co2_captured_edge--retirement_period':              2,
        'edges--co2_captured_edge--existing_capacity':              0,
        'edges--co2_captured_edge--start_vertex':                   start_vtx,
        'edges--co2_storage_edge--end_vertex':                      end_vtx,
    })

injection_df = pd.DataFrame(injection_rows)
out_injection = os.path.join(NOTEBOOK_DIR, 'co2_injection.csv')
injection_df.to_csv(out_injection, index=False)
print(f"Wrote {len(injection_df)} rows -> {out_injection}")
injection_df.head(3)

Wrote 744 rows -> /Users/al3792/Documents_Local/MacroEnergy.jl/ExampleSystems/co2_basin_analysis/co2_injection.csv


,Type,id,location,edges--co2_captured_edge--integer_decisions,edges--co2_captured_edge--can_retire,edges--co2_captured_edge--constraints--CapacityConstraint,edges--co2_captured_edge--unidirectional,edges--co2_captured_edge--commodity,edges--co2_captured_edge--can_expand,edges--co2_captured_edge--uc,...,edges--co2_captured_edge--investment_cost,edges--co2_captured_edge--fixed_om_cost,edges--co2_captured_edge--variable_om_cost,edges--co2_captured_edge--wacc,edges--co2_captured_edge--lifetime,edges--co2_captured_edge--capital_recovery_period,edges--co2_captured_edge--retirement_period,edges--co2_captured_edge--existing_capacity,edges--co2_captured_edge--start_vertex,edges--co2_storage_edge--end_vertex
0,CO2Injection,Region1Beijing_to_Songliao_CO2_Injection,Region1Beijing,FALSE,FALSE,TRUE,TRUE,CO2Captured,TRUE,FALSE,...,157680.0,0,2.93,0.08,40,20,2,0,co2_transported_Region1Beijing_to_Songliao,co2_storage_Songliao
1,CO2Injection,Region1Beijing_to_TurpanHami_CO2_Injection,Region1Beijing,FALSE,FALSE,TRUE,TRUE,CO2Captured,TRUE,FALSE,...,157680.0,0,2.93,0.08,40,20,2,0,co2_transported_Region1Beijing_to_TurpanHami,co2_storage_TurpanHami
2,CO2Injection,Region1Beijing_to_Subei_CO2_Injection,Region1Beijing,FALSE,FALSE,TRUE,TRUE,CO2Captured,TRUE,FALSE,...,157680.0,0,2.93,0.08,40,20,2,0,co2_transported_Region1Beijing_to_Subei,co2_storage_Subei


In [7]:
# Sanity checks
assert len(pipeline_df)  == 31 * 24, f"Expected 744 pipeline rows, got {len(pipeline_df)}"
assert len(injection_df) == 31 * 24, f"Expected 744 injection rows, got {len(injection_df)}"
assert pipeline_df['id'].nunique()  == len(pipeline_df),  "Duplicate pipeline IDs"
assert injection_df['id'].nunique() == len(injection_df), "Duplicate injection IDs"

pipe_ends  = set(pipeline_df['edges--transmission_edge--end_vertex'])
inj_starts = set(injection_df['edges--co2_captured_edge--start_vertex'])
assert pipe_ends == inj_starts, "Pipeline end vertices do not match injection start vertices"

print("All checks passed.")
print(f"Unique storage basins referenced: {injection_df['edges--co2_storage_edge--end_vertex'].nunique()}")
print(f"Variable OM range: {pipeline_df['edges--transmission_edge--variable_om_cost'].min():.2f} "
      f"– {pipeline_df['edges--transmission_edge--variable_om_cost'].max():.2f} $/t")

All checks passed.
Unique storage basins referenced: 24
Variable OM range: 2.29 – 87.76 $/t
